In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
%sql
create table if not exists bankaml.bronze.watermark_control
(
    schema_name string,
    table_name string,
    source string,
    watermark_column string,
    last_processed_value string,
    is_active string,
    updated_timestamp timestamp
)

In [0]:
def get_watermark_column_and_last_processed_value(watermark_df, source, schema, table_name):

    table_watermark_dtl_df = watermark_df.filter(
        (col("source") == source )
        & (col("schema_name") == schema)
        & (col("table_name") == table_name))
    
    watermark_dtl_info = table_watermark_dtl_df.select(
        "watermark_column",
        "last_processed_value"
    ).first()
    
    return watermark_dtl_info["watermark_column"], watermark_dtl_info["last_processed_value"]

In [0]:
def update_last_processed_value(df, source, schema, table_name):

    watermark_table = DeltaTable.forName(spark, "bankaml.bronze.watermark_control")
    watermark_table_df = watermark_table.toDF()
    
    watermark_column, last_processed_value = get_watermark_column_and_last_processed_value(watermark_table_df, source, schema, table_name)
    latest_last_processed_value = df.select(max(col(watermark_column))).first()[0]
    
    if latest_last_processed_value is None:
        return
    
    (
        watermark_table.alias("t")
        .update(
            condition=(
                (col("t.source") == source)
                & (col("t.schema_name") == schema)
                & (col("t.table_name") == table_name)
                & (col("t.watermark_column") == watermark_column)
            ),
            set={
                "last_processed_value": lit(latest_last_processed_value).cast(StringType()),
                "updated_timestamp": current_timestamp()
            }
        )
    )

In [0]:
def get_new_rows_from_source_df(df, source, schema, table_name):

    watermark_table = DeltaTable.forName(spark, "bankaml.bronze.watermark_control")
    watermark_table_df = watermark_table.toDF()

    watermark_column, last_processed_value = get_watermark_column_and_last_processed_value(watermark_table_df, source, schema, table_name)

    if last_processed_value is not None:
        df = df.filter((col(watermark_column)) > (lit(last_processed_value).cast(TimestampType())))

    return df